In [3]:
from collections import deque

def bfs(grafo, inicio):
    visitados = set([inicio])
    cola = deque([inicio])
    orden = []

    while cola:
        nodo = cola.popleft()
        orden.append(nodo)


        for vecino in grafo[nodo]:
            if vecino not in visitados:
                visitados.add(vecino)
                cola.append(vecino)

    return orden



def dfs_camino(grafo, inicio, destino):
    if inicio == destino:
        return [inicio]

    visitados = set([inicio])
    cola = deque([[inicio]])

    while cola:
        camino = cola.popleft()
        nodo = camino[-1]

        for vecino in grafo[nodo]:
            if vecino not in visitados:
                nuevo_camino = camino + [vecino]
                if vecino == destino:
                    return nuevo_camino
                visitados.add(vecino)
                cola.append(nuevo_camino)

    return None


def bfs_niveles(grafo, inicio):
    visitados = set([inicio])
    cola = deque([inicio])
    niveles = []

    while cola:
        nivel_actual = []
        for _ in range(len(cola)):
            nodo = cola.popleft()
            nivel_actual.append(nodo)
            for vecino in grafo[nodo]:
                if vecino not in visitados:
                    visitados.add(vecino)
                    cola.append(vecino)
        niveles.append(nivel_actual)

    return niveles



if __name__ == "__main__":
    grafo = {
        "A": ["B", "C", "D"],
        "B": ["A", "E", "F"],
        "C": ["A", "G"],
        "D": ["A", "G"],
        "E": ["B", "G"],
        "F": ["B", "G"],
        "G": ["C", "D", "E", "F"],
    }


    print("=== BFS básico ===")
    orden = bfs(grafo, "A")
    print("orden de visita: " + " ".join(orden))

    print("\n== camino mas corto")
    camino = dfs_camino(grafo, "A","G")
    print("A G:" + " ".join(camino))

    camino2 = dfs_camino(grafo, "C","E")
    print("c E " + " ".join(camino2))

    print("\n nodos por nivel")
    niveles = bfs_niveles(grafo, "A")
    for i, nivel in enumerate(niveles):
        print(f"Nivel {i}: {nivel}")

=== BFS básico ===
orden de visita: A B C D E F G

== camino mas corto
A G:A C G
c E C G E

 nodos por nivel
Nivel 0: ['A']
Nivel 1: ['B', 'C', 'D']
Nivel 2: ['E', 'F', 'G']


In [5]:
import heapq

def dijkstra(grafo, inicio):
    distancias = {nodo: float('inf') for nodo in grafo}
    distancias[inicio] = 0
    previo = {nodo: None for nodo in grafo}

    heap = [(0, inicio)]

    while heap:
        costo, nodo = heapq.heappop(heap)

        if costo > distancias[nodo]:
            continue

        for vecino, peso in grafo[nodo]:
            nueva_dist = costo + peso
            if nueva_dist < distancias[vecino]:
                distancias[vecino] = nueva_dist
                previo[vecino] = nodo
                heapq.heappush(heap, (nueva_dist, vecino))

    return distancias, previo


def reconstruir_camino(previo, inicio, destino):
    camino = []
    nodo = destino
    while nodo is not None:
        camino.append(nodo)
        nodo = previo[nodo]
    camino.reverse()
    if camino[0] == inicio:
        return camino
    return None


if __name__ == "__main__":
    grafo = {
        "A": [("B", 1), ("C", 4)],
        "B": [("A", 1), ("C", 2), ("D", 5)],
        "C": [("A", 4), ("B", 2), ("D", 1), ("E", 3)],
        "D": [("B", 5), ("C", 1), ("E", 2)],
        "E": [("C", 3), ("D", 2)],
    }


distancias, previo = dijkstra(grafo, 'A')

print("==distancias minimas desde A")
for nodo, dist in distancias.items():
    print(f"A {nodo}: {dist}")


print("\n camino mas cortos")
for destino in grafo:
    camino = reconstruir_camino(previo, "A", destino)
    print(f"A {destino}  {' '.join(camino)}")

==distancias minimas desde A
A A: 0
A B: 1
A C: 3
A D: 4
A E: 6

 camino mas cortos
A A  A
A B  A B
A C  A B C
A D  A B C D
A E  A B C E


In [7]:
from collections import deque

def topological_sort(grafo):
    in_degree = {nodo: 0 for nodo in grafo}
    for nodo in grafo:
        # Ensure we iterate over neighbors that are keys in in_degree
        for vecino, _ in grafo[nodo]:
            if vecino in in_degree:
                in_degree[vecino] += 1

    cola = deque(n for n in in_degree if in_degree[n] == 0)
    orden = []

    while cola:
        nodo = cola.popleft()
        orden.append(nodo)
        for vecino, _ in grafo[nodo]: # Corrected: added space between _ and in
            if vecino in in_degree: # Ensure vecino is in in_degree before decrementing
                in_degree[vecino] -= 1
                if in_degree[vecino] == 0:
                    cola.append(vecino)

    if len(orden) != len(grafo):
        raise ValueError("El grafo tiene ciclos")

    return orden


def dag_shortest_path(grafo, inicio):
    orden = topological_sort(grafo)
    distancias = {nodo: float("inf") for nodo in grafo}
    distancias[inicio] = 0
    previo = {nodo: None for nodo in grafo}

    for nodo in orden:
        if distancias[nodo] == float("inf"):
            continue

        for vecino, peso in grafo[nodo]:
            nueva_dist = distancias[nodo] + peso
            if nueva_dist < distancias[vecino]:
                distancias[vecino] = nueva_dist # Corrected: assignment instead of comparison
                previo[vecino] = nodo

    return distancias, previo


def reconstruir_camino(previo, inicio, destino):
    camino, nodo = [], destino
    while nodo is not None:
        camino.append(nodo)
        nodo = previo[nodo]
    camino.reverse()
    return camino if camino and camino[0] == inicio else None # Added check for empty camino


if __name__ == "__main__":
    grafo = {
        "S": [("A", 2), ("B", 6)],
        "A": [("C", 3), ("D", 1)],
        "B": [("D", 5)],
        "C": [("E", 1)],
        "D": [("E", 4)],
        "E": [],
    }


distancias, previo = dag_shortest_path(grafo, "S")
print("orden topologico")
print(' '.join(topological_sort(grafo))) # Corrected: typo and .join() usage


print("\n distancias minimas desde S")
for nodo, dist in distancias.items():
    d = str(dist) if dist != float("inf") else "-"
    print(f"s {nodo}: {d}") # Corrected: f-string syntax

print("caminos mas cortos")
for destino in grafo:
    camino = reconstruir_camino(previo, "S", destino)
    if camino:
        print(f"s {destino}: {' '.join(camino)}") # Corrected: .join() usage
    else:
        print(f"s {destino}: No alcanzable") # Added a message for unreachable nodes

print("\n pesos negativos no djkstra")
grafo_neg = {
        "S": [("A", 1), ("B", 4)],
        "A": [("C", -2)],
        "B": [("C",  1)],
        "C": [],
    }


dist_neg, prev_neg = dag_shortest_path(grafo_neg, "S") # Corrected: prev,neg to prev_neg
for nodo, dist in dist_neg.items():
    camino = reconstruir_camino(prev_neg, "S", nodo)
    ruta = "- ".join(camino) if camino else "no alcanzable"
    print(f"s {nodo} {dist} ({ruta})")

orden topologico
S A B C D E

 distancias minimas desde S
s S: 0
s A: 2
s B: 6
s C: 5
s D: 3
s E: 6
caminos mas cortos
s S: S
s A: S A
s B: S B
s C: S A C
s D: S A D
s E: S A C E

 pesos negativos no djkstra
s S 0 (S)
s A 1 (S- A)
s B 4 (S- B)
s C -1 (S- A- C)
